## <center>Khalifa University — Agentic AI Program</center>
### <center>Program 5 · Day 1: Advanced Agentic Workflows</center>
# Exercise 1: Sequential Workflow in Plain Python and Ollama

**Module:** Multi-Agent Clinical Reporting Architectures  
**Companion Script:** `Day 1/Excercises/day1_workbook_code/patient_report.py`  
**Estimated Practical Time:** 40 minutes  
**Target Model:** Local Ollama (`qwen3.5:4b`, `qwen3:4b-instruct`, `qwen2.5:7b`, or `qwen2.5:3b`)

---

### Aim & Learning Objectives
By the end of this exercise, you will be able to:
1. **Deconstruct a clinical report task** into three explicit, specialized functional roles: **Extractor**, **Writer**, and **Reviewer**.
2. **Build a deterministic model-call helper** (`ask_model`) using `temperature=0`, local Ollama chat endpoints, and structured JSON enforcement.
3. **Trace data handoffs** between roles using standard Python variables (`note -> facts -> draft_report -> review`).
4. **Implement programmatic validation** to reject malformed review JSON and enforce logical consistency between `decision` and `issues`.
5. **Inject deliberate errors** into the draft to verify that the reviewer catches unsupported clinical claims rather than hallucinating approval.

## System Architecture & Data Flow

This exercise constructs an end-to-end clinical summarization pipeline in pure Python. Instead of asking a single monolithic LLM prompt to read, summarize, and verify a note in one shot, we decompose the task into three decoupled stages:

<p align="center">
  <img src="images/exercise1_pipeline_flow.jpg" alt="Exercise 1: Sequential Clinical Reporting Architecture" style="max-width: 100%; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
</p>

### Cross-Pipeline Data Contract Comparison
| Architectural Dimension | Extractor Output (`facts`) | Writer Output (`draft_report`) | Reviewer Output (`review`) |
|---|---|---|---|
| **Input Data Source** | Raw unstructured note text | Structured facts JSON object | Ground-truth note + candidate draft |
| **Output Data Type** | JSON Object (`dict`) | Unstructured clinical string | JSON Object (`dict`) |
| **Mandatory Schema Keys** | 8 keys: `patient_id`, `age`, `recorded_diagnosis`, `symptoms`, `colonoscopy`, `biopsy`, `stage`, `treatment` | Free-text narrative (<150 words) incorporating all 8 facts | 2 keys: `decision` (string), `issues` (list of strings) |
| **Missing Value Contract** | Exact string `"Not provided"` | Explicitly state `"Not provided"` | Verify `"Not provided"` is preserved without extrapolation |
| **Python Enforcement** | Key existence check (`all(f in facts)`) | Word count check (<150 words) | Strict logic: `PASS` requires `issues == []`, `REVISE` requires `len(issues) > 0` |
| **Failure Mode Caught** | Hallucinated cancer stage or omitted fields | Unchecked medical extrapolation / opinions | Semantic inconsistencies vs original consultation note |

## Prerequisites & Configuration
Before starting, ensure that:
- Ollama is running locally at `http://localhost:11434`.
- A compatible local model (such as `qwen3.5:4b`, `qwen3:4b-instruct`, `qwen2.5:7b`, or `qwen2.5:3b`) is pulled.
- The `ollama` Python library is installed:
  ```bash
  pip install ollama
  ```

In [1]:
import os
import sys

# Model & Ollama configuration
MODEL_NAME = os.environ.get("OLLAMA_MODEL", "qwen3:4b-instruct")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
CONTEXT_TOKENS = 8192
TEMPERATURE = 0

print("Configuration:")
print(f"  Target Model:   {MODEL_NAME}")
print(f"  Ollama URL:     {OLLAMA_URL}")
print(f"  Temperature:    {TEMPERATURE}")
print(f"  Context Tokens: {CONTEXT_TOKENS}")

Configuration:
  Target Model:   qwen3:4b-instruct
  Ollama URL:     http://localhost:11434
  Temperature:    0
  Context Tokens: 8192


## Preflight Check
This preflight cell verifies communication with the local Ollama server, lists available models, and automatically selects the best matching installed model.

In [2]:
import importlib.util
import json
import urllib.request
import urllib.error

class PreflightError(RuntimeError):
    pass

def run_preflight():
    # 1. Check Python library
    if importlib.util.find_spec("ollama") is None:
        raise PreflightError("The 'ollama' library is missing. Install with: pip install ollama")
    
    # 2. Check Ollama API reachability
    try:
        req = urllib.request.Request(f"{OLLAMA_URL}/api/tags")
        with urllib.request.urlopen(req, timeout=3) as resp:
            data = json.loads(resp.read().decode())
            installed_models = [m.get("name") for m in data.get("models", [])]
    except Exception as e:
        raise PreflightError(
            f"Cannot connect to Ollama at {OLLAMA_URL}. Ensure Ollama is running (`ollama serve`). Details: {e}"
        )
    
    print(f"Ollama server is active at {OLLAMA_URL}.")
    print(f"Installed models detected ({len(installed_models)}):")
    for m in installed_models:
        print(f"  - {m}")
    
    # Smart Model Resolution:
    # 1. Respect explicit environment variable if provided
    # 2. Check preferred MODEL_NAME (qwen3:4b-instruct)
    # 3. Check recommended workshop models (qwen3.5:4b, qwen2.5:7b, qwen2.5:3b)
    # 4. Fall back to any installed qwen / available model
    global MODEL_NAME
    env_model = os.environ.get("OLLAMA_MODEL")
    if env_model:
        MODEL_NAME = env_model
        print(f"\nUsing environment specified model: '{MODEL_NAME}'")
        return
    
    if any(m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]) for m in installed_models):
        for m in installed_models:
            if m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]):
                MODEL_NAME = m
                break
        print(f"\nActive model selected: '{MODEL_NAME}' (target match)")
        return
    
    for candidate in ["qwen3.5:4b", "qwen2.5:7b", "qwen2.5:3b"]:
        match = next((m for m in installed_models if m.startswith(candidate) or candidate.startswith(m.split(":")[0])), None)
        if match:
            MODEL_NAME = match
            print(f"\n[NOTICE] '{MODEL_NAME}' selected from installed models (aligns with recommended workshop models).")
            return
            
    if installed_models:
        MODEL_NAME = installed_models[0]
        print(f"\n[NOTICE] Selected first available model: '{MODEL_NAME}'")
    else:
        raise PreflightError("No models found in Ollama. Pull a model with: ollama pull qwen2.5:7b")

try:
    run_preflight()
except PreflightError as err:
    print(f"PREFLIGHT CHECK FAILED: {err}")

Ollama server is active at http://localhost:11434.
Installed models detected (3):
  - qwen3.5:4b
  - qwen3:4b-instruct
  - qwen2.5:3b

Active model selected: 'qwen3:4b-instruct' (target match)


## The Fictional Patient Note (Ground Truth)

Every exercise in this series uses this exact fictional consultation note. 

### Clinical Background & Safety Constraints:
- **Patient Background:** DEMO-001 is a 60-year-old presenting with lower GI symptoms (abdominal pain, hematochezia).
- **Procedures & Pathology:** Colonoscopy identified a mass, and biopsy confirmed adenocarcinoma.
- **Critical Safety Guardrails:** Both **Cancer stage** and **Treatment plan** are explicitly documented as **`Not provided`**.
- **The Core Rule:** A clinical AI assistant must summarize **only** what is documented. In healthcare AI workflows, hallucinating a cancer stage or proposing an unprescribed chemotherapy regimen could lead to catastrophic misguidance. If information is missing, the system must strictly output `"Not provided"`.

In [3]:
PATIENT_NOTE = """
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
""".strip()

print("Loaded Fictional Patient Note (Source of Truth):")
print("-" * 50)
print(PATIENT_NOTE)
print("-" * 50)

Loaded Fictional Patient Note (Source of Truth):
--------------------------------------------------
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
--------------------------------------------------


## Step 1: Create the Model Helper (`ask_model`)

In plain Python, all interactions pass through a centralized helper function.
Key design choices:
- `temperature: 0` ensures deterministic, repeatable output across runs.
- `stream: False` waits for complete generation before returning.
- `CALL_COUNT` keeps an explicit tally of all model requests to track inference budget.
- `json_output=True` requests a JSON object from Ollama.

In [4]:
import json
from ollama import chat

CALL_COUNT = 0

def ask_model(instructions, data, json_output=False):
    """Send one deterministic request to Ollama."""
    global CALL_COUNT
    settings = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user", "content": data},
        ],
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": CONTEXT_TOKENS,
        },
        "stream": False,
    }
    if json_output:
        settings["format"] = "json" if json_output is True else json_output
    CALL_COUNT += 1
    response = chat(**settings)
    return response.message.content

print("ask_model helper initialized.")

ask_model helper initialized.


## Step 2: Role 1 — Fact Extractor (`extract_facts`)

**Extractor Role Responsibility:**
- **Input:** Unstructured consultation note text.
- **Output:** Eight factual fields extracted cleanly into a structured JSON representation.
- **Rules:** Never infer stage or treatment; preserve `"Not provided"` for missing fields.
- **Code Enforcement:** Python code validates that all 8 required keys are present in the resulting dictionary.

In [5]:
EXTRACTION_FIELDS = [
    "patient_id", "age", "recorded_diagnosis", "symptoms",
    "colonoscopy", "biopsy", "stage", "treatment",
]

def extract_facts(note):
    raw = ask_model(
        "Extract exactly these eight fields as a JSON object: patient_id, age, "
        "recorded_diagnosis, symptoms, colonoscopy, biopsy, stage, and treatment. "
        "Use only the note. Use the exact string 'Not provided' for missing facts. "
        "Do not infer stage, explain diagnosis, or recommend treatment. Return JSON only.",
        note,
        json_output=True,
    )
    facts = json.loads(raw)
    missing = [f for f in EXTRACTION_FIELDS if f not in facts]
    if missing:
        raise ValueError(f"Extractor output is missing required fields: {missing}")
    return facts

print("Extracting facts from patient note...")
facts = extract_facts(PATIENT_NOTE)
print(f"Extraction successful! Model requests so far: {CALL_COUNT}")
print("\n--- EXTRACTED FACTS (STRUCTURED JSON) ---")
print(json.dumps(facts, indent=2))

Extracting facts from patient note...
Extraction successful! Model requests so far: 1

--- EXTRACTED FACTS (STRUCTURED JSON) ---
{
  "patient_id": "DEMO-001",
  "age": 60,
  "recorded_diagnosis": "Colon cancer",
  "symptoms": "Abdominal pain and blood in the stool",
  "colonoscopy": "A mass in the colon",
  "biopsy": "Adenocarcinoma",
  "stage": "Not provided",
  "treatment": "Not provided"
}


## Step 3: Role 2 — Report Writer (`write_report`)

**Writer Role Responsibility:**
- **Input:** The extracted JSON facts (never raw unverified data).
- **Output:** A cohesive clinical narrative of under 150 words.
- **Rules:** Say `"recorded diagnosis"`; state when stage and treatment are not provided; do not add medical opinions, prognoses, or causal links.

In [6]:
def write_report(facts):
    facts_text = json.dumps(facts, indent=2)
    return ask_model(
        "Write a patient summary under 150 words. Include all "
        "eight supplied fields. State when stage and treatment "
        "are not provided. Add no conclusions or recommendations. "
        "Say 'recorded diagnosis'; do not infer how or when the "
        "diagnosis was made or connect it causally to symptoms/tests.",
        facts_text,
    )

print("Generating draft clinical report...")
draft_report = write_report(facts)
print(f"Report generated! Model requests so far: {CALL_COUNT}")
print("\n--- DRAFT REPORT ---")
print(draft_report)

Generating draft clinical report...
Report generated! Model requests so far: 2

--- DRAFT REPORT ---
Patient ID: DEMO-001  
Age: 60  
Recorded diagnosis: Colon cancer  
Symptoms: Abdominal pain and blood in the stool  
Colonoscopy: A mass in the colon  
Biopsy: Adenocarcinoma  
Stage: Not provided  
Treatment: Not provided


## Step 4: Role 3 — Reviewer with Schema Validation (`review_report`)

**Reviewer Role Responsibility:**
- **Input:** **Both** the original note (ground truth) and the draft report.
  - *Why both?* Comparing only with extracted facts would fail to catch omissions or errors introduced during extraction! The source note is the single ground-truth authority.
- **Output:** Strictly structured JSON with:
  - `"decision"`: `"PASS"` or `"REVISE"`
  - `"issues"`: list of specific correction strings (empty if PASS, non-empty if REVISE).
- **Code-level checks:** Python explicitly verifies that `decision` matches the `issues` list logically.

In [7]:
def review_report(note, report):
    answer = ask_model(
        "Compare the report with the original note. Check all "
        "eight fields for omissions, errors, and invented facts. "
        "Reject invented explanations of how or when diagnosis was made. "
        "Return JSON with decision (PASS or REVISE) and issues "
        "(a list of specific correction strings). Use PASS only "
        "if there are no issues, with an empty issues list. "
        "Use REVISE with a nonempty issues list otherwise.",
        f"ORIGINAL NOTE:\n{note}\n\nDRAFT:\n{report}",
        json_output=True,
    )
    review = json.loads(answer)
    if not isinstance(review, dict):
        raise ValueError("Review must be a JSON object")
    decision = review.get("decision")
    issues = review.get("issues")
    if decision not in ("PASS", "REVISE"):
        raise ValueError(f"Invalid review decision: {decision}")
    if not isinstance(issues, list) or not all(isinstance(x, str) for x in issues):
        raise ValueError("Issues must be a list of strings")
    if (decision == "PASS" and issues) or (decision == "REVISE" and not issues):
        raise ValueError("Decision and issues disagree: PASS requires [] and REVISE requires nonempty issues.")
    return review

print("Reviewing draft against original note...")
first_review = review_report(PATIENT_NOTE, draft_report)
print(f"Review completed! Model requests so far: {CALL_COUNT}")
print("\n--- REVIEW RESULT ---")
print(json.dumps(first_review, indent=2))

Reviewing draft against original note...
Review completed! Model requests so far: 3

--- REVIEW RESULT ---
{
  "decision": "PASS",
  "issues": []
}


## Step 5: Deliberate Error Injection Test

> **Testing Principle:** Never ask an LLM writer to "make a subtle mistake" and assume it complied. Explicitly inject known, deterministic modifications into the text in Python to create a reproducible test condition.

Here we append two clinically false statements:
1. *"The patient has stage III cancer."* (Direct contradiction of `Cancer stage: Not provided`)
2. *"Chemotherapy is planned."* (Direct contradiction of `Treatment plan: Not provided`)

**Expected Outcome:** `decision: "REVISE"`, with specific issues citing both hallucinations.

In [8]:
# Inject known clinical errors into the clean draft
injected_report = draft_report + (
    "\nThe patient has stage III cancer. Chemotherapy is planned."
)

print("--- INJECTED (FLAWED) REPORT ---")
print(injected_report)

print("\nRunning Reviewer on Flawed Report...")
challenge_review = review_report(PATIENT_NOTE, injected_report)

print("\n--- CHALLENGE REVIEW RESULT ---")
print(json.dumps(challenge_review, indent=2))

# Evaluate detection
if challenge_review["decision"] == "REVISE":
    print("\nSUCCESS: The reviewer caught the injected errors and requested revision!")
    for issue in challenge_review["issues"]:
        print(f"  - Detected issue: {issue}")
else:
    print("\nFAILURE: The reviewer falsely passed an incorrect draft!")

--- INJECTED (FLAWED) REPORT ---
Patient ID: DEMO-001  
Age: 60  
Recorded diagnosis: Colon cancer  
Symptoms: Abdominal pain and blood in the stool  
Colonoscopy: A mass in the colon  
Biopsy: Adenocarcinoma  
Stage: Not provided  
Treatment: Not provided
The patient has stage III cancer. Chemotherapy is planned.

Running Reviewer on Flawed Report...

--- CHALLENGE REVIEW RESULT ---
{
  "decision": "REVISE",
  "issues": [
    "Invented explanation of how or when diagnosis was made: The original note does not state how or when the diagnosis was made. The draft introduces 'The patient has stage III cancer' without any supporting evidence or mention in the original note.",
    "Invented cancer stage: The original note states 'Cancer stage: Not provided'. The draft adds 'The patient has stage III cancer' which is not present in the original and contradicts the original information.",
    "Invented treatment plan: The original note states 'Treatment plan: Not provided'. The draft adds 'Che

## Exercise 1 Checkpoint & Concept Review

Review your outputs and verify your understanding of the architectural decisions:

1. **Why does the reviewer receive the original note as well as the draft?**  
   *Answer:* If an error or omission occurred during extraction in Step 1, comparing the draft only against the extracted facts dictionary would never catch the error. The source note is the single ground-truth authority.

2. **What information is preserved by explicitly writing "Not provided"?**  
   *Answer:* It distinguishes between confirmed absence (e.g., negative test result) and missing/unperformed investigations, preventing downstream roles from assuming or hallucinating clinical values.

3. **Trace data handoffs in plain Python:**
   - `facts`: Python dictionary returned by Extractor, passed directly into Writer.
   - `draft_report`: String formatted by Writer, passed into Reviewer.
   - `review_report(note, report)`: Fresh comparison of ground truth note and candidate draft.

4. **Which checks reject malformed review JSON? Which checks detect an invented clinical statement?**  
   *Answer:* Python code enforces structural schema validation (`isinstance`, key presence, decision/issues agreement), while the LLM reviewer provides semantic factual verification against the source note.